# Phase 2: Local LLM Integration
We will load Llama-3 in 4-bit precision and connect it to our Vector DB.

In [1]:
!pip install bitsandbytes accelerate transformers huggingface_hub langchain langchain-community langchain-huggingface chromadb sentence-transformers

In [1]:
# CELL 2: Login to HuggingFace
# You need a HuggingFace Access Token to download Llama-3.
from huggingface_hub import login
from google.colab import userdata

# Put your HF token in Colab Secrets (the key icon on the left) named 'HF_TOKEN'
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

In [2]:
# CELL 3: Load Local Model in 4-bit
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
)

llm = HuggingFacePipeline(pipeline=text_pipeline)
print("Model loaded successfully in 4-bit mode!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded successfully in 4-bit mode!


In [2]:
!pip install --upgrade --force-reinstall langchain langchain-core langchain-community

  Using cached langchain-1.3.15-py3-none-any.whl.metadata (6.1 kB)
  Using cached langchain_core-1.5.5-py3-none-any.whl.metadata (4.8 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.18-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.11.0-py3-none-any.whl.metadata (22 kB)
  Using cached packaging-26.3-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached uuid_utils-0.17.0-cp312-cp312-ma

In [4]:
# CELL 4: Re-load Vector DB and Create QA Chain (Using modern LCEL)
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Load your Database
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")
vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# 2. Create Prompt
template = """Use the following context to answer the question. If you don't know, just say you don't know.
Context: {context}
Question: {question}
Answer:"""
prompt = PromptTemplate.from_template(template)

# 3. Function to format documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Create the Modern LCEL Chain (Bypasses the buggy langchain.chains module)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Query your Local Llama-3!
response = rag_chain.invoke("What is the summary of the documents?")
print(response)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

/tmp/ipykernel_13712/3393922657.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Use the following context to answer the question. If you don't know, just say you don't know.
Context: 
Question: What is the summary of the documents?
Answer: The documents are a collection of letters, reports, and other records that provide a summary of the events and decisions related to the construction of the new highway. The documents include information about the planning and design of the highway, the selection of contractors and suppliers, and the construction process itself. The documents also include reports on the progress of the construction, as well as any issues or problems that arose during the project. Overall, the documents provide a comprehensive summary of the construction of the new highway. (Don't know) 1/5 (1) 2/5 (2) 3/5 (3) 4/5 (4) 5/5 (5) 6/5 (6) 7/5 (7) 8/5 (8) 9/5 (9) 10/5 (10) 11/5 (11) 12/5 (12) 13/5 (13) 14/5 (14) 15/5 (15) 16/5 (16) 17/5 (17) 18/5 (18) 19/5 (19) 20/5 (20) 21/5 (21) 22/5 (22) 23/5 (23) 24/5 (24) 25/5 (25) 26/5 (26) 27/5 (27) 28/5 (28) 29/